# 🔍 Day-4 · Notebook 03: Hybrid Search (BM25 + Dense) & Re-Ranking
**KIT Campus RAG Agent · Part 3: Overcoming Pure Vector Blindspots on Exact Codes**

Dense semantic search is fantastic for conceptual queries (e.g., *"Can I stay out late if I study in library?"*), but often fails on:
- **Course Codes**: `CS401`, `AI502`, `EC301`
- **Exact Numeric Thresholds**: `₹500`, `7.00 CGPA`, `8:30 PM`, `45% SEE`
- **Acronyms**: `SEE`, `CIA`, `CAT-1`, `HOD`

In this notebook, we implement **Hybrid Search** combining **BM25 Sparse Keyword Retrieval** with **Dense Vector Retrieval** via **Reciprocal Rank Fusion (RRF)**!



In [ ]:
import math
from collections import Counter
from typing import List, Dict

class StudentBM25Retriever:
    """Pure Python BM25 sparse keyword retriever."""
    def __init__(self, corpus: List[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.corpus_size = len(corpus)
        self.tokenized_corpus = [self._tokenize(doc) for doc in corpus]
        self.doc_lens = [len(doc) for doc in self.tokenized_corpus]
        self.avg_doc_len = sum(self.doc_lens) / (self.corpus_size or 1)
        self.doc_freqs = Counter()
        for doc in self.tokenized_corpus:
            self.doc_freqs.update(set(doc))

    def _tokenize(self, text: str) -> List[str]:
        return text.lower().replace("-", " ").replace(".", " ").replace("₹", "rs ").split()

    def get_scores(self, query: str) -> List[float]:
        q_tokens = self._tokenize(query)
        scores = [0.0] * self.corpus_size
        for token in q_tokens:
            if token not in self.doc_freqs:
                continue
            df = self.doc_freqs[token]
            idf = math.log((self.corpus_size - df + 0.5) / (df + 0.5) + 1.0)
            for i, doc in enumerate(self.tokenized_corpus):
                tf = doc.count(token)
                num = tf * (self.k1 + 1.0)
                denom = tf + self.k1 * (1.0 - self.b + self.b * (self.doc_lens[i] / self.avg_doc_len))
                scores[i] += idf * (num / denom)
        return scores

corpus = [
    "Course CS401 Machine Learning & Generative AI taught by Dr. Sundeep K. Exam date is 2026-11-15.",
    "Medical condonation fee is ₹500 per subject for attendance between 65% and 74.9%.",
    "Hostel biometric curfew is strictly 8:30 PM at the lobby turnstiles.",
    "Minimum placement eligibility is 7.00 CGPA with zero standing arrears."
]

bm25 = StudentBM25Retriever(corpus)
query = "What is the exam date for CS401?"
scores = bm25.get_scores(query)

print(f"Query: '{query}'\n")
for doc, score in zip(corpus, scores):
    print(f"[BM25 Score: {score:.3f}] {doc}")



### 2. Reciprocal Rank Fusion (RRF)
How do we merge BM25 scores and Dense Vector scores without calibration errors?
By ranking both lists and calculating the RRF score:
$$\text{RRF\_Score}(d) = \frac{1}{60 + \text{rank}_{\text{dense}}(d)} + \frac{1}{60 + \text{rank}_{\text{bm25}}(d)}$$



In [ ]:
def reciprocal_rank_fusion(dense_ranks: Dict[str, int], bm25_ranks: Dict[str, int], k: int = 60) -> List[tuple]:
    all_keys = set(dense_ranks.keys()).union(set(bm25_ranks.keys()))
    fused_scores = {}
    for key in all_keys:
        d_rank = dense_ranks.get(key, 999)
        b_rank = bm25_ranks.get(key, 999)
        score = (1.0 / (k + d_rank)) + (1.0 / (k + b_rank))
        fused_scores[key] = score
    return sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

# Simulated ranks for 3 candidate chunks
dense_ranks = {"chunk_cs401": 2, "chunk_condonation": 1, "chunk_hostel": 3}
bm25_ranks = {"chunk_cs401": 1, "chunk_condonation": 4, "chunk_hostel": 5}

fused = reciprocal_rank_fusion(dense_ranks, bm25_ranks)
print("Hybrid RRF Ranks:")
for doc_id, score in fused:
    print(f" -> {doc_id}: RRF Score = {score:.5f}")

